# LANCE-seq Figure 4 | Reproducible descriptive analysis

This notebook exposes the complete scientific calculation from spot-level count-like data to the four manuscript panels. It performs descriptive analyses only and does not run differential-expression or significance tests.

### 中文
导入依赖，并集中列出输入、输出、样本顺序、分组、批次映射、颜色以及预定义基因集合。

### English
Import dependencies and declare the input, output, sample order, groups, batch map, colors, and predefined gene sets in one visible configuration block.

In [ ]:
from pathlib import Path
import warnings

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from scipy import sparse
from sklearn.decomposition import PCA

from figure4_utils import (
    EXPECTED_FILES,
    add_descriptive_ellipse,
    configure_style,
    density_values,
    output_inventory,
    prepare_output_dir,
    save_png,
)

INPUT_H5AD = Path("../data/LANCE8.h5ad")
OUTPUT_DIR = Path("outputs")
RANDOM_SEED = 0

SAMPLE_ORDER = ["MN1", "MN5", "CON3", "MN3", "CON1", "MN2", "MN4", "CON2"]
BATCH_TO_SAMPLE = {
    "1": "MN1", "2": "MN2", "3": "MN3", "4": "CON1",
    "5": "MN4", "6": "CON2", "7": "MN5", "8": "CON3",
}
GROUP_ASSIGNMENT = {
    "MN1": "Healthy", "MN5": "Healthy", "CON3": "Healthy",
    "MN3": "Sham", "CON1": "Sham",
    "MN2": "APAP 6 h", "MN4": "APAP 6 h", "CON2": "APAP 6 h",
}
CONDITION_ORDER = ["Healthy", "Sham", "APAP 6 h"]
CONDITION_COLORS = {"Healthy": "#AFC7E8", "Sham": "#A9D5C3", "APAP 6 h": "#E6B0AA"}

MODULE_GENESETS = {
    "Immediate early response": ["Fos", "Fosb", "Jun", "Junb", "Jund", "Egr1", "Egr2", "Atf3", "Dusp1", "Ier2", "Nr4a1"],
    "General cellular stress": ["Atf3", "Ddit3", "Atf4", "Hspa1a", "Hspa1b", "Hsp90aa1", "Dusp1", "Gadd45a", "Gadd45b"],
    "Inflammatory / NF-kB response": ["Tnf", "Il1b", "Nfkbia", "Nfkb1", "Rela", "Ccl2", "Cxcl1", "Cxcl2", "Ptgs2", "Icam1", "Socs3"],
    "Oxidative stress / NRF2": ["Hmox1", "Nqo1", "Gclc", "Gclm", "Srxn1", "Txnrd1", "Slc7a11", "Gsta1", "Gsta2", "Gstp1"],
    "Glutathione metabolism": ["Gclc", "Gclm", "Gss", "Gsr", "Gpx1", "Gpx2", "Gsta1", "Gsta2", "Gstm1", "Gstm2", "Gstp1", "Slc7a11"],
    "Cell death / injury response": ["Ddit3", "Atf4", "Gadd45a", "Gadd45b", "Gadd45g", "Bbc3", "Bax", "Bcl2l11", "Casp3", "Fas", "Trp53inp1"],
}
MODULE_ORDER = list(MODULE_GENESETS)
APAP_MODULE_ORDER = MODULE_ORDER[2:]
MARKER_GENES = ["Hmox1", "Nqo1", "Atf3", "Ddit3", "Fgf21", "Gdf15", "Cyp2e1"]

configure_style()
OUTPUT_DIR = prepare_output_dir(OUTPUT_DIR)
np.random.seed(RANDOM_SEED)
print(f"Input: {INPUT_H5AD}")
print(f"Output: {OUTPUT_DIR.resolve()}")

### 中文
直接读取 h5ad，显示批次到样本的映射；保留至少在 3 个 spot 中表达的基因，并确认 `X` 为非负、近似整数的 count-like 数据。

### English
Read the h5ad directly, show the batch-to-sample mapping, retain genes detected in at least three spots, and confirm that `X` is nonnegative count-like data.

In [2]:
if not INPUT_H5AD.is_file():
    raise FileNotFoundError(f"Input h5ad not found: {INPUT_H5AD}")

adata = sc.read_h5ad(INPUT_H5AD)
adata.var_names_make_unique()
if "batch" not in adata.obs:
    raise ValueError("adata.obs['batch'] is required")

batches = adata.obs["batch"].astype(str).str.strip()
observed_batches = set(batches.unique())
if observed_batches != set(BATCH_TO_SAMPLE):
    raise ValueError(f"Unexpected batches: {sorted(observed_batches)}")
sample_labels = batches.map(BATCH_TO_SAMPLE)
if sample_labels.isna().any() or set(sample_labels.unique()) != set(SAMPLE_ORDER):
    raise ValueError("The expected eight samples were not recovered")
adata.obs["sample"] = sample_labels.to_numpy()

genes_before_filter = int(adata.n_vars)
sc.pp.filter_genes(adata, min_cells=3)
genes_after_spot_filter = int(adata.n_vars)

counts = adata.X
nonzero_values = counts.data if sparse.issparse(counts) else np.asarray(counts).ravel()
if nonzero_values.size == 0 or not np.all(np.isfinite(nonzero_values)) or np.min(nonzero_values) < 0:
    raise ValueError("Figure 4 requires finite, nonnegative count-like X values")
integer_like_fraction = float(np.isclose(nonzero_values, np.rint(nonzero_values), atol=1e-8, rtol=0).mean())
if integer_like_fraction < 0.995:
    raise ValueError("X is not sufficiently count-like for pseudobulk aggregation")
if integer_like_fraction < 1.0:
    warnings.warn(f"X is strongly count-like ({integer_like_fraction:.6f} integer-like nonzero fraction); non-integer weights are retained")

print(f"adata: {adata.n_obs:,} spots × {genes_before_filter:,} genes")
print(f"genes after min_cells=3: {genes_after_spot_filter:,}")
print(pd.DataFrame({"batch": list(BATCH_TO_SAMPLE), "sample": list(BATCH_TO_SAMPLE.values())}).to_string(index=False))

adata: 12,473 spots × 24,082 genes
genes after min_cells=3: 24,082
batch sample
    1    MN1
    2    MN2
    3    MN3
    4   CON1
    5    MN4
    6   CON2
    7    MN5
    8   CON3


cell_4:30: UserWarning: X is strongly count-like (0.997764 integer-like nonzero fraction); non-integer weights are retained


### 中文
按样本直接汇总 spot counts，计算 CPM；保留至少 2 个样本中 CPM ≥ 1 的基因，并转换为 `log2(CPM + 1)`。

### English
Sum spot counts within each sample, calculate CPM, retain genes with CPM ≥ 1 in at least two samples, and transform to `log2(CPM + 1)`.

In [3]:
sample_vector = adata.obs["sample"].to_numpy()
sample_count_rows = []
spot_counts = {}
for sample in SAMPLE_ORDER:
    mask = sample_vector == sample
    spot_counts[sample] = int(mask.sum())
    if not mask.any():
        raise ValueError(f"No spots found for {sample}")
    sample_count_rows.append(np.asarray(counts[mask].sum(axis=0)).ravel())

pseudobulk_counts = pd.DataFrame(
    np.vstack(sample_count_rows),
    index=SAMPLE_ORDER,
    columns=adata.var_names.astype(str),
)
library_sizes = pseudobulk_counts.sum(axis=1)
if (library_sizes <= 0).any():
    raise ValueError("A pseudobulk sample has zero library size")

cpm_all = pseudobulk_counts.div(library_sizes, axis=0) * 1_000_000
gene_keep = (cpm_all >= 1.0).sum(axis=0) >= 2
pseudobulk_logcpm = np.log2(cpm_all.loc[:, gene_keep] + 1)

pseudobulk_summary = pd.DataFrame({
    "group": [GROUP_ASSIGNMENT[s] for s in SAMPLE_ORDER],
    "spots": [spot_counts[s] for s in SAMPLE_ORDER],
    "library_size": library_sizes.loc[SAMPLE_ORDER].astype(float),
}, index=SAMPLE_ORDER)
print(f"genes after CPM filter: {pseudobulk_logcpm.shape[1]:,}")
print(pseudobulk_summary.to_string())

genes after CPM filter: 15,534
         group  spots  library_size
MN1    Healthy   1226    54683904.5
MN5    Healthy    948    36443214.0
CON3   Healthy   1929    68011619.0
MN3       Sham   1680    53471196.0
CON1      Sham   2034    56000013.0
MN2   APAP 6 h   1845    41758691.0
MN4   APAP 6 h    932    37394588.0
CON2  APAP 6 h   1879    40864969.0


### 中文
在 8 个样本 × CPM 过滤后基因的 `log2(CPM + 1)` 矩阵上运行居中、不额外标准化的 PCA，并直接绘制 Fig. 4f。

### English
Run centered, unscaled PCA on the 8-sample × CPM-filtered-gene `log2(CPM + 1)` matrix and draw Fig. 4f directly.

In [4]:
n_components = min(7, pseudobulk_logcpm.shape[0] - 1, pseudobulk_logcpm.shape[1])
pca = PCA(n_components=n_components, random_state=RANDOM_SEED)
pca_values = pca.fit_transform(pseudobulk_logcpm.to_numpy())
pca_coordinates = pd.DataFrame(
    pca_values,
    index=SAMPLE_ORDER,
    columns=[f"PC{i}" for i in range(1, n_components + 1)],
)
explained_variance = pca.explained_variance_ratio_.copy()

fig, ax = plt.subplots(figsize=(7.2, 5.7))
x_span = float(np.ptp(pca_coordinates.loc[SAMPLE_ORDER, "PC1"]))
y_span = float(np.ptp(pca_coordinates.loc[SAMPLE_ORDER, "PC2"]))
for condition in CONDITION_ORDER:
    members = [s for s in SAMPLE_ORDER if GROUP_ASSIGNMENT[s] == condition]
    points = pca_coordinates.loc[members, ["PC1", "PC2"]].to_numpy()
    add_descriptive_ellipse(ax, points, CONDITION_COLORS[condition], x_span, y_span)

label_specs = {
    "MN1": (5, 5, "left", "bottom"), "MN5": (-5, -7, "right", "top"),
    "CON3": (5, 5, "left", "bottom"), "MN3": (-5, 5, "right", "bottom"),
    "CON1": (5, -5, "left", "top"), "MN2": (-5, 5, "right", "bottom"),
    "MN4": (-5, -7, "right", "top"), "CON2": (5, 5, "left", "bottom"),
}
for sample in SAMPLE_ORDER:
    condition = GROUP_ASSIGNMENT[sample]
    marker = "o" if sample.startswith("MN") else "s"
    x_value, y_value = pca_coordinates.loc[sample, ["PC1", "PC2"]]
    ax.scatter(x_value, y_value, s=92, marker=marker, facecolor=CONDITION_COLORS[condition], edgecolor="black", linewidth=0.7, zorder=3)
    dx, dy, ha, va = label_specs[sample]
    ax.annotate(sample, (x_value, y_value), xytext=(dx, dy), textcoords="offset points", fontsize=8.5, ha=ha, va=va, zorder=4)
ax.axhline(0, color="#D9D9D9", linewidth=0.6, zorder=0)
ax.axvline(0, color="#D9D9D9", linewidth=0.6, zorder=0)
ax.set_xlabel(f"PC1 ({explained_variance[0] * 100:.1f}%)")
ax.set_ylabel(f"PC2 ({explained_variance[1] * 100:.1f}%)")
handles = [
    Line2D([0], [0], marker="o", linestyle="none", label=condition, markerfacecolor=CONDITION_COLORS[condition], markeredgecolor="black", markersize=7)
    for condition in CONDITION_ORDER
] + [
    Line2D([0], [0], marker=marker, linestyle="none", label=platform, markerfacecolor="white", markeredgecolor="black", markersize=7)
    for platform, marker in (("MN", "o"), ("CON", "s"))
]
ax.legend(handles=handles, frameon=False, fontsize=8, loc="upper left", bbox_to_anchor=(1.01, 1.0))
sns.despine(ax=ax)
fig.tight_layout()
figure4f_path = save_png(fig, OUTPUT_DIR, EXPECTED_FILES[0])

print("Explained variance (%):", np.round(explained_variance[:3] * 100, 3).tolist())
print(pca_coordinates[["PC1", "PC2"]].round(3).to_string())
print(figure4f_path)

Explained variance (%): [46.345, 33.102, 11.621]
         PC1     PC2
MN1   32.900  53.799
MN5   42.487  11.895
CON3  43.422  17.685
MN3    7.704 -37.466
CON1  22.696 -64.187
MN2  -62.864  20.029
MN4  -35.549  -8.388
CON2 -50.797   6.633
outputs\Fig4_5A_Pseudobulk_PCA_PC1_PC2.png


### 中文
将每个基因在 8 个样本间进行 z-score 标准化（总体标准差，`ddof=0`），再对每个预定义模块中可用基因取均值，直接绘制 Fig. 4g 六模块热图。

### English
Z-score each gene across the eight samples using the population standard deviation (`ddof=0`), average available genes within each predefined module, and draw the six-module Fig. 4g heatmap directly.

In [5]:
gene_means = pseudobulk_logcpm.mean(axis=0)
gene_sds = pseudobulk_logcpm.std(axis=0, ddof=0)
variable_genes = gene_sds > 0
gene_zscores = (pseudobulk_logcpm.loc[:, variable_genes] - gene_means.loc[variable_genes]).div(gene_sds.loc[variable_genes], axis=1)

module_scores = pd.DataFrame(index=MODULE_ORDER, columns=SAMPLE_ORDER, dtype=float)
coverage_rows = []
for module in MODULE_ORDER:
    requested_genes = MODULE_GENESETS[module]
    matched_genes = [gene for gene in requested_genes if gene in gene_zscores.columns]
    missing_genes = [gene for gene in requested_genes if gene not in gene_zscores.columns]
    if len(matched_genes) < 4 or len(matched_genes) / len(requested_genes) < 0.25:
        raise ValueError(f"Insufficient gene coverage for module: {module}")
    module_scores.loc[module] = gene_zscores.loc[:, matched_genes].mean(axis=1)
    coverage_rows.append({
        "module": module,
        "requested": len(requested_genes),
        "matched": len(matched_genes),
        "matched_genes": ";".join(matched_genes),
        "missing_genes": ";".join(missing_genes),
    })
module_coverage = pd.DataFrame(coverage_rows)

heatmap_data = module_scores.loc[MODULE_ORDER, SAMPLE_ORDER].astype(float)
vmax = float(np.max(np.abs(heatmap_data.to_numpy())))
fig = plt.figure(figsize=(8.8, 5.25))
grid = fig.add_gridspec(nrows=2, ncols=2, height_ratios=[0.18, 4.2], width_ratios=[20, 0.8], hspace=0.08, wspace=0.20)
ax_condition = fig.add_subplot(grid[0, 0])
condition_codes = np.array([[CONDITION_ORDER.index(GROUP_ASSIGNMENT[sample]) for sample in SAMPLE_ORDER]])
condition_cmap = ListedColormap([CONDITION_COLORS[c] for c in CONDITION_ORDER])
ax_condition.imshow(condition_codes, aspect="auto", cmap=condition_cmap, vmin=-0.5, vmax=2.5)
ax_condition.set_yticks([0], ["Condition"], fontsize=8)
ax_condition.set_xticks([])
ax_condition.tick_params(length=0)
for x_pos, label in ((1.0, "Healthy"), (3.5, "Sham"), (6.0, "APAP 6 h")):
    ax_condition.text(x_pos, 0, label, ha="center", va="center", fontsize=7.5)
for spine in ax_condition.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(0.5)
    spine.set_color("black")

ax = fig.add_subplot(grid[1, 0])
cax = fig.add_subplot(grid[1, 1])
sns.heatmap(heatmap_data, ax=ax, cmap="RdBu_r", center=0, vmin=-vmax, vmax=vmax, linewidths=0.35, linecolor="white", cbar=True, cbar_ax=cax, cbar_kws={"label": "Mean gene z-score"}, xticklabels=SAMPLE_ORDER, yticklabels=MODULE_ORDER)
ax.axhline(3, color="black", linewidth=0.8)
ax.set_xlabel("")
ax.set_ylabel("")
ax.tick_params(axis="x", rotation=45, labelsize=8, colors="black")
ax.tick_params(axis="y", rotation=0, labelsize=8, colors="black")
fig.subplots_adjust(left=0.30, right=0.91, bottom=0.19, top=0.97, hspace=0.08, wspace=0.20)
figure4g_path = save_png(fig, OUTPUT_DIR, EXPECTED_FILES[1])

print(module_coverage[["module", "requested", "matched", "missing_genes"]].to_string(index=False))
print(module_scores.round(3).to_string())
print(figure4g_path)

                       module  requested  matched missing_genes
     Immediate early response         11       11              
      General cellular stress          9        9              
Inflammatory / NF-kB response         11        9     Tnf;Ptgs2
      Oxidative stress / NRF2         10       10              
       Glutathione metabolism         12       11          Gpx2
 Cell death / injury response         11       11              
                                 MN1    MN5   CON3    MN3   CON1    MN2    MN4   CON2
Immediate early response      -1.087 -1.054 -1.003 -0.183 -0.241  1.181  1.261  1.124
General cellular stress       -0.381 -1.106 -0.974  0.136  0.115  0.597  1.034  0.580
Inflammatory / NF-kB response -0.687 -0.876 -0.752 -0.517 -0.701  1.415  1.029  1.087
Oxidative stress / NRF2        0.012 -0.708 -0.630 -0.284 -0.651  0.952  0.683  0.626
Glutathione metabolism         0.431 -0.531 -0.518 -0.025 -0.299  0.249  0.419  0.275
Cell death / injury response  -0.511

### 中文
对 CPM 过滤后的全部基因直接计算 MN2/MN4 平均表达与差值：`mean = (MN2 + MN4) / 2`，`difference = MN2 − MN4`。仅标注七个固定代表基因并绘制描述性 Fig. 4h，不进行 DEG 或显著性检验。

### English
For every CPM-filtered gene, calculate `mean = (MN2 + MN4) / 2` and `difference = MN2 − MN4`. Label the seven fixed representative genes and draw descriptive Fig. 4h without DEG or significance testing.

In [6]:
agreement_table = pd.DataFrame({
    "gene": pseudobulk_logcpm.columns,
    "MN2_logcpm": pseudobulk_logcpm.loc["MN2"].to_numpy(float),
    "MN4_logcpm": pseudobulk_logcpm.loc["MN4"].to_numpy(float),
})
agreement_table["mean_expression"] = (agreement_table["MN2_logcpm"] + agreement_table["MN4_logcpm"]) / 2
agreement_table["MN2_minus_MN4"] = agreement_table["MN2_logcpm"] - agreement_table["MN4_logcpm"]
agreement_table["absolute_difference"] = agreement_table["MN2_minus_MN4"].abs()

x_values = agreement_table["mean_expression"].to_numpy(float)
differences = agreement_table["MN2_minus_MN4"].to_numpy(float)
median_difference = float(np.median(differences))
q005, q995 = np.quantile(differences, [0.005, 0.995])
robust_span = float(q995 - q005)
initial_limits = (float(q005 - robust_span * 0.035), float(q995 + robust_span * 0.035))
marker_differences = agreement_table.loc[agreement_table["gene"].isin(MARKER_GENES), "MN2_minus_MN4"]
if marker_differences.empty:
    raise ValueError("None of the fixed marker genes is available")
initial_span = initial_limits[1] - initial_limits[0]
display_limits = (
    min(initial_limits[0], float(marker_differences.min()) - 0.03 * initial_span),
    max(initial_limits[1], float(marker_differences.max()) + 0.03 * initial_span),
)

marker_rows = []
for gene in MARKER_GENES:
    row = agreement_table.loc[agreement_table["gene"] == gene]
    if row.empty:
        warnings.warn(f"Gene {gene} is unavailable; its label is skipped")
        marker_rows.append({"gene": gene, "available": False})
        continue
    value = row.iloc[0]
    marker_rows.append({
        "gene": gene,
        "available": True,
        "MN2_logcpm": float(value["MN2_logcpm"]),
        "MN4_logcpm": float(value["MN4_logcpm"]),
        "mean_expression": float(value["mean_expression"]),
        "MN2_minus_MN4": float(value["MN2_minus_MN4"]),
    })
marker_table = pd.DataFrame(marker_rows)

density, density_norm, density_cmap = density_values(x_values, differences)
draw_order = np.argsort(density)
fig, ax = plt.subplots(figsize=(6.8, 5.6))
scatter = ax.scatter(x_values[draw_order], differences[draw_order], c=density[draw_order], cmap=density_cmap, norm=density_norm, s=6, alpha=0.43, edgecolors="none")
ax.axhline(0, color="#333333", linestyle="--", linewidth=1.0)
ax.axhline(median_difference, color="#9B5C57", linewidth=0.75)
ax.set_ylim(*display_limits)
ax.set_xlabel("Mean log2(CPM + 1) expression\n(MN2 and MN4)")
ax.set_ylabel("MN2 − MN4 expression difference")
ax.text(0.02, 0.98, f"Median Δ = {median_difference:+.3f}", transform=ax.transAxes, ha="left", va="top", fontsize=8, color="#6B4743")
ax.text(0.985, 0.96, "Positive: higher in MN2\nPrior 0 h biopsy", transform=ax.transAxes, ha="right", va="top", fontsize=7)
ax.text(0.985, 0.04, "Negative: higher in MN4\nNo prior 0 h biopsy", transform=ax.transAxes, ha="right", va="bottom", fontsize=7)
label_offsets = {
    "Hmox1": (-10, 20, "right"), "Nqo1": (-8, 16, "right"),
    "Atf3": (-10, 16, "right"), "Ddit3": (-10, -18, "right"),
    "Fgf21": (8, 14, "left"), "Gdf15": (10, 16, "left"),
    "Cyp2e1": (8, 14, "left"),
}
for row in marker_table.itertuples(index=False):
    if not row.available:
        continue
    ax.scatter(row.mean_expression, row.MN2_minus_MN4, s=38, facecolor="white", edgecolor="black", linewidth=0.9, zorder=5)
    dx, dy, ha = label_offsets[row.gene]
    ax.annotate(row.gene, xy=(row.mean_expression, row.MN2_minus_MN4), xytext=(dx, dy), textcoords="offset points", ha=ha, va="center", fontsize=8.5, color="black", arrowprops={"arrowstyle": "-", "color": "#555555", "linewidth": 0.5, "shrinkA": 1.5, "shrinkB": 3}, clip_on=True, zorder=6)
ax.spines[["top", "right"]].set_visible(False)
colorbar = fig.colorbar(scatter, ax=ax, fraction=0.025, pad=0.025, aspect=35)
colorbar.set_label("Gene density", fontsize=7)
colorbar.ax.tick_params(labelsize=6, width=0.6, length=2)
fig.tight_layout()
figure4h_path = save_png(fig, OUTPUT_DIR, EXPECTED_FILES[2])

descriptive_summary = pd.Series({
    "n_genes": len(agreement_table),
    "median_difference": np.median(differences),
    "mean_difference": np.mean(differences),
    "median_absolute_difference": np.median(np.abs(differences)),
})
print(descriptive_summary.to_string())
print(marker_table.round(6).to_string(index=False))
print(figure4h_path)

n_genes                       15534.000000
median_difference                 0.039642
mean_difference                   0.147107
median_absolute_difference        0.227897
  gene  available  MN2_logcpm  MN4_logcpm  mean_expression  MN2_minus_MN4
 Hmox1       True    9.613738    9.466517         9.540127       0.147221
  Nqo1       True    3.412874    3.134529         3.273702       0.278345
  Atf3       True    8.075268    7.816969         7.946119       0.258299
 Ddit3       True    6.812523    7.137568         6.975046      -0.325045
 Fgf21       True    8.956176    7.886116         8.421146       1.070060
 Gdf15       True   10.335511   10.163408        10.249460       0.172104
Cyp2e1       True    9.608879   10.885958        10.247418      -1.277079
outputs\Fig4_6A3_MN2_MN4_expression_agreement_labeled.png


### 中文
直接从 Fig. 4g 的同一模块分数中提取 MN2 与 MN4，计算 `MN2 − MN4`，并绘制四个 APAP 损伤相关模块的描述性 Fig. 4i；连线和差值不表示统计推断。

### English
Extract MN2 and MN4 from the same module scores used in Fig. 4g, calculate `MN2 − MN4`, and draw descriptive Fig. 4i for four APAP injury-related modules; lines and differences are not inferential statistics.

In [7]:
module_comparison = pd.DataFrame({
    "module": APAP_MODULE_ORDER,
    "MN2_score": [float(module_scores.loc[module, "MN2"]) for module in APAP_MODULE_ORDER],
    "MN4_score": [float(module_scores.loc[module, "MN4"]) for module in APAP_MODULE_ORDER],
})
module_comparison["MN2_minus_MN4"] = module_comparison["MN2_score"] - module_comparison["MN4_score"]
module_comparison["absolute_difference"] = module_comparison["MN2_minus_MN4"].abs()
module_comparison["direction"] = np.where(module_comparison["MN2_minus_MN4"] > 0, "MN2 higher", "MN4 higher")

plot_data = module_comparison.set_index("module").loc[APAP_MODULE_ORDER].reset_index()
y_positions = np.arange(len(APAP_MODULE_ORDER))[::-1]
fig, ax = plt.subplots(figsize=(7.7, 4.5))
for y_pos, row in zip(y_positions, plot_data.itertuples(index=False)):
    ax.plot([row.MN2_score, row.MN4_score], [y_pos, y_pos], color="#C9C9C9", linewidth=0.8, zorder=1)
    ax.scatter(row.MN2_score, y_pos, s=76, facecolor="#E6B0AA", edgecolor="#6E514E", linewidth=0.75, zorder=3)
    ax.scatter(row.MN4_score, y_pos, s=76, facecolor="white", edgecolor="#E6B0AA", linewidth=1.35, zorder=3)
    ax.text(1.012, y_pos, f"{row.MN2_minus_MN4:+.2f}".replace("-", "−"), transform=ax.get_yaxis_transform(), ha="left", va="center", fontsize=8.0, color="black", clip_on=False)
ax.axvline(0, color="#D9D9D9", linewidth=0.7)
ax.set_yticks(y_positions, APAP_MODULE_ORDER)
ax.set_xlabel("Relative module score")
ax.set_ylabel("")
ax.text(1.012, 1.18, "Δ (MN2 − MN4)", transform=ax.transAxes, ha="left", va="bottom", fontsize=8.5, clip_on=False)
ax.text(1.012, 1.09, "Positive: higher in MN2\nNegative: higher in MN4", transform=ax.transAxes, ha="left", va="top", fontsize=6.0, color="#444444", clip_on=False)
handles = [
    Line2D([0], [0], marker="o", linestyle="none", label="MN2  Prior 0 h biopsy", markerfacecolor="#E6B0AA", markeredgecolor="#6E514E", markersize=7),
    Line2D([0], [0], marker="o", linestyle="none", label="MN4  No prior 0 h biopsy", markerfacecolor="white", markeredgecolor="#E6B0AA", markersize=7),
]
ax.legend(handles=handles, frameon=False, loc="lower right", fontsize=8)
for side in ("top", "right", "left"):
    ax.spines[side].set_visible(False)
ax.tick_params(axis="y", length=0)
x_min = min(plot_data["MN2_score"].min(), plot_data["MN4_score"].min())
x_max = max(plot_data["MN2_score"].max(), plot_data["MN4_score"].max())
comparison_span = max(x_max - x_min, 0.5)
ax.set_xlim(x_min - 0.18 * comparison_span, x_max + 0.18 * comparison_span)
fig.subplots_adjust(left=0.33, right=0.83, bottom=0.18, top=0.82)
figure4i_path = save_png(fig, OUTPUT_DIR, EXPECTED_FILES[3])

print(module_comparison.round(6).to_string(index=False))
print(figure4i_path)

                       module  MN2_score  MN4_score  MN2_minus_MN4  absolute_difference  direction
Inflammatory / NF-kB response   1.415331   1.029453       0.385879             0.385879 MN2 higher
      Oxidative stress / NRF2   0.952463   0.683035       0.269427             0.269427 MN2 higher
       Glutathione metabolism   0.249118   0.419170      -0.170053             0.170053 MN4 higher
 Cell death / injury response   0.423257   0.702286      -0.279029             0.279029 MN4 higher
outputs\Fig4_6B3_APAP_injury_module_comparison.png


### 中文
汇总最终四张 PNG。Notebook 不生成 Figure 4e 图件。

### English
Summarize the four final PNG files. This notebook does not generate a Figure 4e image.

In [8]:
final_outputs = output_inventory(OUTPUT_DIR)
print(final_outputs.to_string(index=False))

                                         filename   bytes
               Fig4_5A_Pseudobulk_PCA_PC1_PC2.png  349642
                      Fig4_5B_6module_heatmap.png  265733
Fig4_6A3_MN2_MN4_expression_agreement_labeled.png 2262358
       Fig4_6B3_APAP_injury_module_comparison.png  273789
